# US College Salaries — Region × School Type
**Python · pandas · scikit-learn · matplotlib**

**Objective:** Explore associations between college-level starting pay, mid-career pay, geographic region, and school type. Compare salary outcomes against region-wide averages and evaluate a predictive model on schools held out from training.

**Important:** These are historical, college-level aggregates from the WSJ / PayScale dataset, **not current job-market salaries**, individual outcomes, or causal effects of attending a school. The data snapshot date is not established.

**Reproducibility:** Run the cells from top to bottom. The notebook uses local `salaries-by-college-type.csv` and `salaries-by-region.csv` files when available, otherwise downloads them from an attributed public GitHub copy. Internet access is required for this fallback.

## 0) Environment

Install dependencies in a terminal before opening the notebook:

```bash
python -m pip install -r requirements.txt
jupyter notebook college_salary_final_project.ipynb
```

Run the notebook from its repository folder. All notebook execution outputs have been cleared because the analysis was revised; **execute the notebook to generate fresh results**.

In [ ]:
# Verify required libraries without modifying the environment.
import pandas as pd
import numpy as np
import matplotlib
import sklearn

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("matplotlib:", matplotlib.__version__)
print("scikit-learn:", sklearn.__version__)

## 1) Setup & Data Load

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Local files take priority. If absent, download the public historical WSJ / PayScale copies.
DATA_FILES = {
    "college": "salaries-by-college-type.csv",
    "region": "salaries-by-region.csv",
}
DATA_BASE_URL = (
    "https://raw.githubusercontent.com/UBC-MDS-2018-19/"
    "DSCI_522_Salary-vs-College/master/data/raw_data/"
)

def load_salary_data(filename):
    path = Path(filename)
    source = str(path) if path.is_file() else DATA_BASE_URL + filename
    try:
        table = pd.read_csv(source, na_values=["N/A", "NA", ""])
    except Exception as exc:
        raise RuntimeError(
            f"Unable to load {filename}. Place it beside this notebook or "
            f"check internet access to {source}"
        ) from exc
    print(f"Loaded {filename}: {len(table)} rows from {source}")
    return table

df_type_raw = load_salary_data(DATA_FILES["college"])
df_region_raw = load_salary_data(DATA_FILES["region"])

print("College-type and region tables (first five rows each):")
display(df_type_raw.head())
display(df_region_raw.head())

## 2) Schema Check & Cleaning

In [ ]:
# Schema validation, cleaning, and currency parsing.
dft = df_type_raw.copy()
dfr = df_region_raw.copy()

def normalize_cols(frame):
    frame = frame.copy()
    frame.columns = (frame.columns.str.strip()
                     .str.replace(r"[^A-Za-z0-9]+", "_", regex=True)
                     .str.strip("_"))
    return frame

dft = normalize_cols(dft)
dfr = normalize_cols(dfr)
for frame_name, frame, required in [
    ("college type", dft, {"School_Name", "School_Type",
                            "Starting_Median_Salary", "Mid_Career_Median_Salary"}),
    ("region", dfr, {"School_Name", "Region",
                      "Starting_Median_Salary", "Mid_Career_Median_Salary"}),
]:
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Missing columns in {frame_name} data: {sorted(missing)}")

salary_cols_t = [c for c in dft.columns if "Salary" in c]
salary_cols_r = [c for c in dfr.columns if "Salary" in c]

def clean_money(frame, columns):
    for col in columns:
        values = frame[col].astype("string").str.replace(r"[$,]", "", regex=True)
        frame[col] = pd.to_numeric(values, errors="coerce")
    return frame

dft = clean_money(dft, salary_cols_t)
dfr = clean_money(dfr, salary_cols_r)
dft["School_Name"] = dft["School_Name"].astype("string").str.strip()
dfr["School_Name"] = dfr["School_Name"].astype("string").str.strip()
print("Missing values in college-type data:")
display(dft.isna().sum())
print("Missing values in region data:")
display(dfr.isna().sum())

## 3) Merge Region Label + Compute Region Baselines

In [ ]:
# Attach each school's geographical region to its school-type record.
region_conflicts = dfr.groupby("School_Name")["Region"].nunique(dropna=True)
if (region_conflicts > 1).any():
    raise ValueError("Conflicting region labels for some school names; review the source data.")
region_lookup = dfr[["School_Name", "Region"]].drop_duplicates(subset=["School_Name"])
df = dft.merge(region_lookup, on="School_Name", how="left", validate="many_to_one")

# Descriptive region averages are NOT used as predictive features in the model below.
region_baseline = (
    dfr.groupby("Region")[salary_cols_r].mean()
       .reset_index()
       .rename(columns={c: c + "_RegionAvg" for c in salary_cols_r})
)
df = df.merge(region_baseline, on="Region", how="left", validate="many_to_one")

missing_region = int(df["Region"].isna().sum())
print(f"College rows: {len(df)}. Without a region match: {missing_region}.")
print("Joined data (first five rows):")
display(df.head())

## 4) Descriptive Statistics

In [ ]:
college_salary_cols = [c for c in df.columns if ('Salary' in c) and (not c.endswith('_RegionAvg'))]
summary = df[college_salary_cols].describe().T
print('Title: Descriptive stats (college-level salaries)')
print('Caption: Count, mean, std, min, quartiles, max for college salary fields.')
display(summary)

## 5) Uplift vs Region Averages

In [ ]:
# Required columns for uplift
need = ['Starting_Median_Salary', 'Mid_Career_Median_Salary',
        'Starting_Median_Salary_RegionAvg', 'Mid_Career_Median_Salary_RegionAvg']
missing = [c for c in need if c not in df.columns]
assert not missing, f"Missing columns for uplift: {missing}"

df['Uplift_Starting']  = df['Starting_Median_Salary']   - df['Starting_Median_Salary_RegionAvg']
df['Uplift_MidCareer'] = df['Mid_Career_Median_Salary'] - df['Mid_Career_Median_Salary_RegionAvg']

print('Title: Uplift preview')
print('Caption: School salaries minus their region’s averages.')
df_display = df[['School_Name', 'Region', 'School_Type',
                 'Starting_Median_Salary', 'Starting_Median_Salary_RegionAvg',
                 'Uplift_Starting', 'Mid_Career_Median_Salary',
                 'Mid_Career_Median_Salary_RegionAvg', 'Uplift_MidCareer']].head(10)
df_display.index = range(1, len(df_display) + 1)
display(df_display)

## 6) Visualization — Top 10 Schools by Mid‑Career Uplift

In [ ]:
top10 = df.sort_values('Uplift_MidCareer', ascending=False).head(10)

plt.figure(figsize=(10, 6))
plt.barh(top10['School_Name'][::-1], top10['Uplift_MidCareer'][::-1])
plt.title('Figure 1. Top 10 Schools by Mid‑Career Uplift over Region Average')
plt.xlabel('Uplift (USD)')
plt.ylabel('School Name')
plt.tight_layout()
plt.show()
print('Caption: Mid‑career salary minus **regional average** mid‑career salary (higher is better).')

## 7) Region × School Type — Group Means & Plot

In [ ]:
target = 'Starting_Median_Salary'
if all(c in df.columns for c in ['Region','School_Type',target]):
    pivot = df.pivot_table(values=target, index='Region', columns='School_Type', aggfunc='mean')
    print('Title: Group means of Starting Salary by Region × School Type')
    print('Caption: Interaction-like differences across region and school type.')
    display(pivot)

    # Clustered bars (matplotlib)
    regions = pivot.index.tolist()
    types = pivot.columns.tolist()
    x = np.arange(len(regions))
    width = 0.8 / max(1, len(types))

    plt.figure(figsize=(12, 6))
    for i, t in enumerate(types):
        plt.bar(x + i*width, pivot[t].to_numpy(dtype=float, na_value=np.nan), width=width, label=str(t))
    plt.xticks(x + (len(types)-1)*width/2, regions, rotation=45, ha='right')
    plt.title('Figure 2. Starting Salary by Region × School Type')
    plt.xlabel('Region')
    plt.ylabel('Starting Median Salary (USD)')
    plt.legend(title='School Type')
    plt.tight_layout()
    plt.show()
    print('Caption: Clustered bars highlight joint differences by region and type.')
else:
    print('Required columns missing for Region × School Type analysis.')

## 8) Correlation & predictive modeling with a held-out test set

The prior notebook reported only **in-sample R²**, which does not assess generalization. Here we split schools **before fitting preprocessors or models**, compare a starting-salary-only baseline with a regularized model that includes region, school type, and their interaction, and report held-out R² and MAE.

The combined `Region_Type` categorical variable encodes a Region × School Type interaction. Ridge regression controls instability from overlapping categories and a small dataset. The held-out test set is used once for evaluation; no hyperparameters are tuned on it.

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_absolute_error

# Descriptive correlation only: no predictive or causal inference here.
correlation = df[["Starting_Median_Salary", "Mid_Career_Median_Salary"]].corr()
print("Pearson correlation (college-level summaries):")
display(correlation)

target = "Mid_Career_Median_Salary"
predictors = ["Starting_Median_Salary", "Region", "School_Type"]
model_data = df[["School_Name", *predictors, target]].dropna().copy()
if len(model_data) < 40:
    raise ValueError("Too few complete college records for a meaningful train/test split.")
print(f"Complete records for modeling: {len(model_data)}")

# All splitting happens before fitting any estimator or encoder.
train_df, test_df = train_test_split(model_data, test_size=0.25,
                                    random_state=42, shuffle=True)

def model_inputs(frame):
    features = frame[predictors].copy()
    features["Region_Type"] = (
        features["Region"].astype(str) + " | " + features["School_Type"].astype(str)
    )
    return features

X_train = model_inputs(train_df)
X_test = model_inputs(test_df)
y_train, y_test = train_df[target], test_df[target]

baseline = Pipeline([
    ("preprocess", ColumnTransformer([
        ("starting_salary", StandardScaler(), ["Starting_Median_Salary"])
    ])),
    ("regressor", LinearRegression())
])
interaction = Pipeline([
    ("preprocess", ColumnTransformer([
        ("starting_salary", StandardScaler(), ["Starting_Median_Salary"]),
        ("categories", OneHotEncoder(handle_unknown="ignore"),
         ["Region", "School_Type", "Region_Type"])
    ])),
    ("regressor", Ridge(alpha=5.0))
])

metrics = []
for name, estimator in [
    ("Starting salary baseline", baseline),
    ("Region × School Type + Ridge", interaction),
]:
    estimator.fit(X_train, y_train)
    pred_train = estimator.predict(X_train)
    pred_test = estimator.predict(X_test)
    metrics.append({
        "Model": name,
        "Train R2": r2_score(y_train, pred_train),
        "Test R2": r2_score(y_test, pred_test),
        "Test MAE (USD)": mean_absolute_error(y_test, pred_test),
    })

results = pd.DataFrame(metrics)
print("Held-out evaluation (higher test R² and lower test MAE are better):")
display(results)

# Cross-validation only on training data; pipeline refits preprocessing per fold.
folds = KFold(n_splits=5, shuffle=True, random_state=42)
cv_r2 = cross_val_score(interaction, X_train, y_train, cv=folds, scoring="r2")
print(f"Training-only 5-fold CV R²: {cv_r2.mean():.3f} +/- {cv_r2.std():.3f}")
print(f"Training size = {len(train_df)}, held-out test size = {len(test_df)}")

test_predictions = interaction.predict(X_test)
bounds = [
    min(float(y_test.min()), float(test_predictions.min())),
    max(float(y_test.max()), float(test_predictions.max()))
]
plt.figure(figsize=(7, 5))
plt.scatter(y_test, test_predictions, alpha=0.75)
plt.plot(bounds, bounds, linestyle="--", color="gray", label="Ideal prediction")
plt.title("Figure 3. Actual vs Predicted (Held-Out Schools)")
plt.xlabel("Observed mid-career median salary (USD)")
plt.ylabel("Predicted mid-career median salary (USD)")
plt.legend()
plt.tight_layout()
plt.show()

# These estimates describe historical college-level relationships, not individuals.

## 9) Uplift Distribution

In [ ]:
plt.figure(figsize=(8,5))
vals = df['Uplift_MidCareer'].dropna().values
plt.hist(vals, bins=20)
plt.title('Figure 4. Distribution of Mid‑Career Uplift over Region Average')
plt.xlabel('Uplift (USD)')
plt.ylabel('Count of Schools')
plt.tight_layout()
plt.show()
print('Caption: Spread of schools above/below their region’s baseline.')

## 10) Save Artifacts

In [ ]:
from pathlib import Path

# Save derived descriptive data; avoid committing generated outputs.
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
output_path = output_dir / "college_salary_master_with_regionavg.csv"
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} college rows to {output_path}")

## 11) Interpretation and limitations

- **Descriptive uplift:** Compare each school's starting and mid-career median salary with the average for its region. These are historical descriptive differences, not school effects.
- **Model comparison:** Use the **held-out test R² and MAE printed above**, rather than the previously saved in-sample R², to assess whether region and school type add predictive value beyond starting salary.
- **Scope:** These are school-level aggregates; they do not measure an individual graduate's expected salary. Major, experience, student selection, graduating cohort, costs, and other confounders are not controlled.
- **Data age:** The original WSJ/PayScale data is historical and its exact snapshot date is unverified; **do not use its dollar values as contemporary pay estimates**.
- **Limitations:** Small and potentially unrepresentative sample, incomplete school-name matches, and potentially rare Region × Type combinations. Test R² may vary with the random split; repeated independent validation would be needed for robust deployment claims.

The output cells are intentionally cleared after refactoring. Execute **Run All** to produce current numerical findings.